# Align2DBulk — all features
Comprehensive demo of every Align2DBulk parameter: basic alignment, max_shift clamping, binning, crop vs no-crop, non-zero reference, summary/offsets, Show3D browsing, and mean-frame SNR improvement.

In [ ]:
try:
    %load_ext autoreload
    %autoreload 2
    %env ANYWIDGET_HMR=1
except Exception:
    pass  # autoreload unavailable (Colab Python 3.12+)

In [ ]:
import json

import numpy as np
import torch
import quantem.widget
from quantem.widget import Align2DBulk, Show2D, Show3D, profile

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
rng = np.random.default_rng(42)
print(f"Device: {device}")
print(f"quantem.widget {quantem.widget.__version__}")
profile()

## Generate synthetic STEM frames
Create a perovskite crystal reference (SrTiO3-like [001] HAADF-STEM) and 49 shifted copies with independent scan noise. Shifts are sub-pixel, drawn uniformly from [-8, 8] px.

In [ ]:
N = 256
rows = torch.arange(N, device=device, dtype=torch.float32)
yy, xx = torch.meshgrid(rows, rows, indexing="ij")
img = torch.zeros((N, N), device=device, dtype=torch.float32)
a = 16
sig_a, sig_b = 2.2, 1.8
ni = N // a + 3
ai_idx = torch.arange(-1, ni, device=device, dtype=torch.float32)
# A-site columns
a_cx = (ai_idx * a).unsqueeze(1)
a_cy = (ai_idx * a).unsqueeze(0)
a_cx_flat = a_cx.expand(ni + 1, ni + 1).reshape(-1, 1, 1)
a_cy_flat = a_cy.expand(ni + 1, ni + 1).reshape(-1, 1, 1)
img += (torch.exp(-((xx.unsqueeze(0) - a_cx_flat)**2 + (yy.unsqueeze(0) - a_cy_flat)**2) / (2 * sig_a**2))).sum(0)
# B-site columns
b_cx = (ai_idx * a + a / 2).unsqueeze(1)
b_cy = (ai_idx * a + a / 2).unsqueeze(0)
b_cx_flat = b_cx.expand(ni + 1, ni + 1).reshape(-1, 1, 1)
b_cy_flat = b_cy.expand(ni + 1, ni + 1).reshape(-1, 1, 1)
img += 0.55 * (torch.exp(-((xx.unsqueeze(0) - b_cx_flat)**2 + (yy.unsqueeze(0) - b_cy_flat)**2) / (2 * sig_b**2))).sum(0)
ref = img.cpu().numpy().astype(np.float32)
print(f"Reference image: {ref.shape}, range [{ref.min():.3f}, {ref.max():.3f}]")

In [ ]:
def fourier_shift(img, dy, dx):
    h, w = img.shape
    fy = np.fft.fftfreq(h).reshape(-1, 1)
    fx = np.fft.fftfreq(w).reshape(1, -1)
    phase = np.exp(-2j * np.pi * (fy * dy + fx * dx))
    return np.real(np.fft.ifft2(np.fft.fft2(img) * phase)).astype(np.float32)

n_frames = 50
true_shifts = [(0.0, 0.0)] + [(rng.uniform(-8, 8), rng.uniform(-8, 8)) for _ in range(n_frames - 1)]
frames = []
for dy, dx in true_shifts:
    frame = fourier_shift(ref, dy, dx)
    frame += rng.normal(0, 0.03, (N, N)).astype(np.float32)
    frames.append(frame)
stack = np.stack(frames, axis=0)
print(f"Stack: {stack.shape}")
print(f"First 5 true shifts (dy, dx):")
for i, (dy, dx) in enumerate(true_shifts[:5]):
    print(f"  [{i}] dy={dy:+6.2f}  dx={dx:+6.2f}")
print(f"  ... ({n_frames - 5} more frames)")

## 1. Basic alignment (reference=0)
Align all 50 frames to frame 0 via GPU-accelerated FFT phase correlation with DFT sub-pixel refinement. Default settings: `crop=True`, `bin=1`, `max_shift=15`.

In [ ]:
Show2D([stack[0], stack[10], stack[30]], labels=["Frame 0 (ref)", "Frame 10", "Frame 30"], title="Raw frames (before alignment)")

In [ ]:
bulk = Align2DBulk(stack, reference=0, max_shift=15, title="Perovskite alignment")
bulk

In [ ]:
print(f"Aligned shape: {bulk.stack.shape}")
print(f"Reference frame: {bulk.reference_idx}")
print(f"NCC range: [{min(bulk.ncc):.4f}, {max(bulk.ncc):.4f}]")
print(f"Crop box: {bulk.crop_box}")

## 2. max_shift clamping
When `max_shift` is smaller than the actual shift, offsets get clamped. Here we create frames with shifts up to 25 px but limit the search to 5 px.

In [ ]:
# Create frames with large shifts (up to 25 px)
large_shifts = [(0.0, 0.0)] + [(rng.uniform(-25, 25), rng.uniform(-25, 25)) for _ in range(9)]
large_frames = []
for dy, dx in large_shifts:
    frame = fourier_shift(ref, dy, dx)
    frame += rng.normal(0, 0.03, (N, N)).astype(np.float32)
    large_frames.append(frame)
large_stack = np.stack(large_frames, axis=0)
# Align with max_shift=5 (clamped)
bulk_clamped = Align2DBulk(large_stack, reference=0, max_shift=5, crop=False, title="Clamped (max_shift=5)")
# Align with max_shift=30 (unclamped)
bulk_unclamped = Align2DBulk(large_stack, reference=0, max_shift=30, crop=False, title="Unclamped (max_shift=30)")
print(f"{'Frame':>5}  {'True dx':>8}  {'Clamp dx':>9}  {'Full dx':>8}  {'True dy':>8}  {'Clamp dy':>9}  {'Full dy':>8}")
print("-" * 70)
for i, (dy, dx) in enumerate(large_shifts):
    c_dx, c_dy = bulk_clamped.offsets[i]
    u_dx, u_dy = bulk_unclamped.offsets[i]
    print(f"{i:>5}  {dx:>+8.2f}  {c_dx:>+9.2f}  {u_dx:>+8.2f}  {dy:>+8.2f}  {c_dy:>+9.2f}  {u_dy:>+8.2f}")
# Check that clamped offsets are within bounds
max_clamped_dx = max(abs(o[0]) for o in bulk_clamped.offsets)
max_clamped_dy = max(abs(o[1]) for o in bulk_clamped.offsets)
print(f"\nMax |dx| clamped: {max_clamped_dx:.2f} (limit=5)")
print(f"Max |dy| clamped: {max_clamped_dy:.2f} (limit=5)")

## 3. bin parameter
The `bin` parameter applies avg_pool2d binning on GPU after alignment. Output dimensions are reduced by the bin factor.

In [ ]:
bulk_bin1 = Align2DBulk(stack, reference=0, max_shift=15, bin=1, crop=False, title="bin=1")
bulk_bin2 = Align2DBulk(stack, reference=0, max_shift=15, bin=2, crop=False, title="bin=2")
bulk_bin4 = Align2DBulk(stack, reference=0, max_shift=15, bin=4, crop=False, title="bin=4")
print(f"bin=1 shape: {bulk_bin1.stack.shape}")
print(f"bin=2 shape: {bulk_bin2.stack.shape}")
print(f"bin=4 shape: {bulk_bin4.stack.shape}")

In [ ]:
Show2D(
    [bulk_bin1.stack[0], bulk_bin2.stack[0], bulk_bin4.stack[0]],
    labels=[f"bin=1 ({bulk_bin1.stack.shape[1]}x{bulk_bin1.stack.shape[2]})",
            f"bin=2 ({bulk_bin2.stack.shape[1]}x{bulk_bin2.stack.shape[2]})",
            f"bin=4 ({bulk_bin4.stack.shape[1]}x{bulk_bin4.stack.shape[2]})"],
    title="Binning comparison (frame 0)",
)

## 4. crop=True vs crop=False
With `crop=True` (default), the output is trimmed to the common overlap region. With `crop=False`, shifted regions are zero-padded and full dimensions are kept.

In [ ]:
bulk_crop = Align2DBulk(stack, reference=0, max_shift=15, crop=True, title="crop=True")
bulk_nocrop = Align2DBulk(stack, reference=0, max_shift=15, crop=False, title="crop=False")
print(f"crop=True  shape: {bulk_crop.stack.shape}  crop_box: {bulk_crop.crop_box}")
print(f"crop=False shape: {bulk_nocrop.stack.shape}  crop_box: {bulk_nocrop.crop_box}")

In [ ]:
Show2D(
    [bulk_crop.stack[0], bulk_nocrop.stack[0]],
    labels=[f"crop=True ({bulk_crop.stack.shape[1]}x{bulk_crop.stack.shape[2]})",
            f"crop=False ({bulk_nocrop.stack.shape[1]}x{bulk_nocrop.stack.shape[2]})"],
    title="Crop comparison (frame 0)",
)

## 5. Non-zero reference frame
Align to a middle frame (`reference=25`) instead of frame 0. The middle frame's offset is always (0, 0) and NCC is 1.0.

In [ ]:
bulk_mid = Align2DBulk(stack, reference=25, max_shift=15, title="Reference = frame 25")
bulk_mid

In [ ]:
print(f"Reference index: {bulk_mid.reference_idx}")
print(f"Offset of reference frame: {bulk_mid.offsets[25]}")
print(f"NCC of reference frame: {bulk_mid.ncc[25]:.4f}")
print(f"Aligned shape: {bulk_mid.stack.shape}")

## 6. Summary and offsets
Use `.summary()` for a human-readable table. Access raw data via `.offsets`, `.ncc`, `.offsets_json`, and `.crop_box`.

In [ ]:
bulk.summary()

In [ ]:
# offsets_json is a JSON string synced to the widget frontend
entries = json.loads(bulk.offsets_json)
print(f"offsets_json entries: {len(entries)}")
print(f"First entry: {entries[0]}")
print(f"Last entry:  {entries[-1]}")

In [ ]:
# Access raw lists
print(f"Offsets (first 5): {bulk.offsets[:5]}")
print(f"NCC (first 5):     {[f'{v:.4f}' for v in bulk.ncc[:5]]}")
print(f"Crop box:          {bulk.crop_box}")
print(f"Mean NCC:          {np.mean(bulk.ncc):.4f}")

## 7. Show3D of aligned stack
Browse through all 50 aligned frames interactively.

In [ ]:
Show3D(bulk.stack, title="Aligned stack (50 frames)")

## 8. Mean of aligned stack
Averaging over aligned frames improves SNR by ~sqrt(N). Compare a single frame to the 50-frame average.

In [ ]:
mean_img = bulk.stack.mean(axis=0)
single_frame = bulk.stack[0]
print(f"Single frame std: {single_frame.std():.4f}")
print(f"Mean image std:   {mean_img.std():.4f}")
print(f"SNR improvement:  ~{np.sqrt(bulk.stack.shape[0]):.1f}x (sqrt({bulk.stack.shape[0]}))")

In [ ]:
Show2D(
    [single_frame, mean_img],
    labels=["Single frame", f"Mean of {bulk.stack.shape[0]} frames"],
    title="SNR improvement from frame averaging",
)